# Order Items table: cleaning

Run from the folder that contains raw/ and cleaned/ (your VS Code workspace root).
Needs cleaned/orders_clean.csv (from the orders notebook) for the join check.

Decisions and findings for this table are written up in `cleaning_notes/README_order_items.md`.

### 1. Imports and settings

In [ ]:
import pandas as pd
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

RAW = Path('raw')
CLEAN = Path('cleaned')
CLEAN.mkdir(exist_ok=True)

### 2. Load the raw order items file

In [ ]:
order_items = pd.read_csv(RAW / 'olist_order_items_dataset.csv')

### 3. First inspection: structure, nulls, preview

In [ ]:
order_items.info()
print(order_items.isna().sum())     # expect all zeros
print(order_items.head())

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  str    
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  str    
 3   seller_id            112650 non-null  str    
 4   shipping_limit_date  112650 non-null  str    
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), str(4)
memory usage: 6.0 MB
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64
                           order_id  order_item_id  \
0  00010242fe8c5a6d1ba2dd792cb16214              1   
1  00018f77f2f0320c557190d7a144bdd3              1   
2  000229ec398224ef6ca0657da4fc703

### 4. Convert the shipping deadline from text to datetime

In [ ]:
order_items['shipping_limit_date'] = pd.to_datetime(order_items['shipping_limit_date'])
print(order_items['shipping_limit_date'].dtype)     # datetime64[us]

datetime64[us]


### 5. Key check: one row per item, so the key is the PAIR (order_id, order_item_id)

In [ ]:
print(order_items.duplicated(subset=['order_id', 'order_item_id']).sum())   # expect 0
print(order_items['order_id'].nunique())            # 98,666 orders have items
print(order_items['order_item_id'].max())           # 21 items in the largest order

# Item numbers should run 1..n inside each order with no gaps
g = order_items.groupby('order_id')['order_item_id'].agg(['max', 'count'])
print((g['max'] != g['count']).sum())               # expect 0

0
98666
21
0


### 6. Join integrity with orders

In [ ]:
orders = pd.read_csv(CLEAN / 'orders_clean.csv', usecols=['order_id', 'order_status'])

print(order_items['order_id'].isin(orders['order_id']).value_counts())   # True 112,650 (no orphan items)
print(orders['order_id'].isin(order_items['order_id']).value_counts())   # True 98,666 / False 775

# Which orders have no items?
no_items = ~orders['order_id'].isin(order_items['order_id'])
print(orders.loc[no_items, 'order_status'].value_counts())
# unavailable 603, canceled 164, created 5, invoiced 2, shipped 1

order_id
True    112650
Name: count, dtype: int64
order_id
True     98666
False      775
Name: count, dtype: int64
order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64


### 7. Money columns

In [ ]:
print(order_items[['price', 'freight_value']].describe())
print((order_items['price'] <= 0).sum())                              # 0
print((order_items['freight_value'] == 0).sum())                      # 383 free-shipping rows
print((order_items['freight_value'] > order_items['price']).sum())    # 4,124 rows
print(order_items.loc[order_items['price'] == order_items['price'].max()])   # the 6,735 item

               price  freight_value
count  112650.000000  112650.000000
mean      120.653739      19.990320
std       183.633928      15.806405
min         0.850000       0.000000
25%        39.900000      13.080000
50%        74.990000      16.260000
75%       134.900000      21.150000
max      6735.000000     409.680000
0
383
4124
                              order_id  order_item_id  \
3556  0812eb902a67711a1cb742b3cdaa65ae              1   

                            product_id                         seller_id  \
3556  489ae2aa008f021502940f251d4cce7f  e3b4998c7a498169dc7bce44e6bb6277   

     shipping_limit_date   price  freight_value  
3556 2017-02-16 20:37:36  6735.0         194.31  


### 8. Engineered column: what the customer paid per item

In [ ]:
order_items['item_total'] = order_items['price'] + order_items['freight_value']

### 9. Save the cleaned table and verify the file on disk

In [ ]:
order_items.to_csv(CLEAN / 'order_items_clean.csv', index=False)

check = pd.read_csv(CLEAN / 'order_items_clean.csv', parse_dates=['shipping_limit_date'])
print(check.shape)                                                    # expect (112650, 8)
print(check.dtypes)                                                   # shipping_limit_date: datetime64
print(check.duplicated(subset=['order_id', 'order_item_id']).sum())   # expect 0
print(check['item_total'].sum().round(2))                             # 15,843,553.24

(112650, 8)
order_id                          str
order_item_id                   int64
product_id                        str
seller_id                         str
shipping_limit_date    datetime64[us]
price                         float64
freight_value                 float64
item_total                    float64
dtype: object
0
15843553.24
